# Embed the ICD-10 chapter descriptions with Octen-Embedding-8B

Generated by `scripts/build_icd_notebook.py`. Edit the generator, not this file.

Run on RTX Pro 6000 with Internet OFF. Attach `boinhbo/medxir-package-octen8b`.


In [ ]:
import glob
import json
import os
import time

MODEL = 'Octen/Octen-Embedding-8B'
MODEL_DIR = 'Octen-Embedding-8B'
DOC_PREFIX = ' '   # Octen's document prompt, as in the corpus embedding
EXPECT_GPU = '6000'
BATCH = 16
MAX_LENGTH = 1024
OUT = '/kaggle/working'
ROWS = [
 {
  "id": "I",
  "lang": "vi",
  "text": "Bệnh nhiễm trùng và ký sinh trùng: nhiễm khuẩn, nhiễm virus, nấm, ký sinh trùng, giun sán. Lao phổi, lao hạch, HIV/AIDS, viêm gan virus B, viêm gan C, sốt xuất huyết dengue, sởi, thủy đậu, quai bị, tay chân miệng, cúm, thương hàn, tả, lỵ, tiêu chảy nhiễm khuẩn, giang mai, lậu, bệnh lây truyền qua đường tình dục, nấm da, nấm candida, sốt rét, sán lá gan, giun đũa, nhiễm trùng huyết, uốn ván, dại, zona thần kinh, mụn cóc, HPV, herpes. Kháng sinh, thuốc kháng virus, tiêm vắc xin phòng bệnh truyền nhiễm."
 },
 {
  "id": "I",
  "lang": "en",
  "text": "Certain infectious and parasitic diseases: bacterial, viral, fungal and parasitic infections. Tuberculosis, HIV/AIDS, viral hepatitis B and C, dengue fever, measles, chickenpox, mumps, hand-foot-mouth disease, influenza, typhoid, cholera, dysentery, infectious diarrhoea, syphilis, gonorrhoea, sexually transmitted infections, fungal skin infection, candidiasis, malaria, helminths, worms, sepsis, tetanus, rabies, herpes zoster, warts, HPV, herpes simplex. Antibiotics, antiviral drugs, vaccination against infectious diseases."
 },
 {
  "id": "I",
  "lang": "zh",
  "text": "某些传染病和寄生虫病：细菌、病毒、真菌和寄生虫感染。结核病、肺结核、艾滋病、乙型肝炎、丙型肝炎、登革热、麻疹、水痘、腮腺炎、手足口病、流感、伤寒、霍乱、痢疾、感染性腹泻、梅毒、淋病、性传播疾病、真菌感染、念珠菌、疟疾、寄生虫、蛔虫、败血症、破伤风、狂犬病、带状疱疹、疣、HPV、疱疹。抗生素、抗病毒药、疫苗接种。"
 },
 {
  "id": "II",
  "lang": "vi",
  "text": "Khối u, ung thư và u lành tính: ung thư phổi, ung thư gan, ung thư dạ dày, ung thư đại trực tràng, ung thư vú, ung thư cổ tử cung, ung thư buồng trứng, ung thư tuyến tiền liệt, ung thư tuyến giáp, ung thư vòm họng, ung thư máu, bệnh bạch cầu, u lympho, u não, u xơ tử cung, u nang, polyp, u mỡ, u máu, khối u ác tính, di căn, giai đoạn ung thư, hóa trị, xạ trị, phẫu thuật cắt khối u, sinh thiết, chất chỉ điểm khối u, tầm soát ung thư."
 },
 {
  "id": "II",
  "lang": "en",
  "text": "Neoplasms: cancer and benign tumours. Lung cancer, liver cancer, stomach cancer, colorectal cancer, breast cancer, cervical cancer, ovarian cancer, prostate cancer, thyroid cancer, nasopharyngeal carcinoma, leukaemia, lymphoma, brain tumour, uterine fibroids, cyst, polyp, lipoma, haemangioma, malignant tumour, metastasis, cancer stage, chemotherapy, radiotherapy, tumour resection, biopsy, tumour markers, cancer screening."
 },
 {
  "id": "II",
  "lang": "zh",
  "text": "肿瘤：癌症和良性肿瘤。肺癌、肝癌、胃癌、结直肠癌、乳腺癌、宫颈癌、卵巢癌、前列腺癌、甲状腺癌、鼻咽癌、白血病、淋巴瘤、脑瘤、子宫肌瘤、囊肿、息肉、脂肪瘤、血管瘤、恶性肿瘤、转移、癌症分期、化疗、放疗、肿瘤切除、活检、肿瘤标志物、癌症筛查。"
 },
 {
  "id": "III",
  "lang": "vi",
  "text": "Bệnh của máu, cơ quan tạo máu và rối loạn miễn dịch: thiếu máu, thiếu máu thiếu sắt, thiếu máu tán huyết, thalassemia, tan máu bẩm sinh, hemophilia, rối loạn đông máu, giảm tiểu cầu, xuất huyết giảm tiểu cầu, giảm bạch cầu, suy tủy xương, lách to, suy giảm miễn dịch, bệnh sarcoidosis. Xét nghiệm công thức máu, huyết sắc tố, ferritin, truyền máu, bổ sung sắt."
 },
 {
  "id": "III",
  "lang": "en",
  "text": "Diseases of the blood and blood-forming organs and immune disorders: anaemia, iron deficiency anaemia, haemolytic anaemia, thalassaemia, haemophilia, coagulation disorders, thrombocytopenia, thrombocytopenic purpura, neutropenia, leukopenia, bone marrow failure, aplastic anaemia, splenomegaly, immunodeficiency, sarcoidosis. Complete blood count, haemoglobin, ferritin, blood transfusion, iron supplements."
 },
 {
  "id": "III",
  "lang": "zh",
  "text": "血液及造血器官疾病和免疫疾病：贫血、缺铁性贫血、溶血性贫血、地中海贫血、血友病、凝血障碍、血小板减少、血小板减少性紫癜、白细胞减少、再生障碍性贫血、骨髓衰竭、脾大、免疫缺陷、结节病。血常规、血红蛋白、铁蛋白、输血、补铁。"
 },
 {
  "id": "IV",
  "lang": "vi",
  "text": "Bệnh nội tiết, dinh dưỡng và chuyển hóa: đái tháo đường, tiểu đường type 1, type 2, đường huyết cao, hạ đường huyết, HbA1c, insulin, bệnh tuyến giáp, cường giáp, Basedow, suy giáp, bướu cổ, nhân giáp, viêm tuyến giáp, rối loạn mỡ máu, cholesterol cao, triglyceride, béo phì, thừa cân, suy dinh dưỡng, thiếu vitamin D, thiếu canxi, còi xương, gout, acid uric máu cao, bệnh tuyến thượng thận, hội chứng Cushing, rối loạn hormone, buồng trứng đa nang, mất nước, rối loạn điện giải."
 },
 {
  "id": "IV",
  "lang": "en",
  "text": "Endocrine, nutritional and metabolic diseases: diabetes mellitus type 1 and type 2, high blood sugar, hypoglycaemia, HbA1c, insulin, thyroid disease, hyperthyroidism, Graves disease, hypothyroidism, goitre, thyroid nodules, thyroiditis, dyslipidaemia, high cholesterol, triglycerides, obesity, overweight, malnutrition, vitamin D deficiency, calcium deficiency, rickets, gout, hyperuricaemia, adrenal disease, Cushing syndrome, hormone disorders, polycystic ovary syndrome, dehydration, electrolyte imbalance."
 },
 {
  "id": "IV",
  "lang": "zh",
  "text": "内分泌、营养和代谢疾病：糖尿病、1型糖尿病、2型糖尿病、高血糖、低血糖、糖化血红蛋白、胰岛素、甲状腺疾病、甲亢、甲减、甲状腺肿、甲状腺结节、甲状腺炎、血脂异常、高胆固醇、甘油三酯、肥胖、超重、营养不良、维生素D缺乏、缺钙、佝偻病、痛风、高尿酸血症、肾上腺疾病、库欣综合征、激素紊乱、多囊卵巢综合征、脱水、电解质紊乱。"
 },
 {
  "id": "V",
  "lang": "vi",
  "text": "Rối loạn tâm thần và hành vi: trầm cảm, rối loạn lo âu, căng thẳng, stress, mất ngủ, rối loạn giấc ngủ, rối loạn lưỡng cực, tâm thần phân liệt, rối loạn hoảng sợ, ám ảnh cưỡng chế, sang chấn tâm lý, sa sút trí tuệ, Alzheimer, tự kỷ, tăng động giảm chú ý ADHD, chậm phát triển trí tuệ, rối loạn ăn uống, nghiện rượu, nghiện ma túy, nghiện thuốc lá, ý định tự tử, tâm lý trẻ em, liệu pháp tâm lý, thuốc chống trầm cảm."
 },
 {
  "id": "V",
  "lang": "en",
  "text": "Mental and behavioural disorders: depression, anxiety disorder, stress, insomnia, sleep disorders, bipolar disorder, schizophrenia, panic disorder, obsessive-compulsive disorder, trauma, PTSD, dementia, Alzheimer disease, autism, ADHD, intellectual disability, eating disorders, alcohol addiction, drug addiction, smoking dependence, suicidal thoughts, child psychology, psychotherapy, antidepressants."
 },
 {
  "id": "V",
  "lang": "zh",
  "text": "精神和行为障碍：抑郁症、焦虑症、压力、失眠、睡眠障碍、双相情感障碍、精神分裂症、惊恐障碍、强迫症、心理创伤、痴呆、阿尔茨海默病、自闭症、多动症、智力障碍、进食障碍、酒精依赖、药物成瘾、烟草依赖、自杀念头、儿童心理、心理治疗、抗抑郁药。"
 },
 {
  "id": "VI",
  "lang": "vi",
  "text": "Bệnh hệ thần kinh: đau đầu, đau nửa đầu migraine, động kinh, co giật, Parkinson, đa xơ cứng, viêm màng não, viêm não, liệt mặt, liệt dây thần kinh số 7, đau dây thần kinh tọa, đau dây thần kinh, bệnh lý thần kinh ngoại biên, tê bì tay chân, hội chứng ống cổ tay, rối loạn tiền đình, chóng mặt, nhược cơ, bại não, rối loạn giấc ngủ do thần kinh, ngưng thở khi ngủ, thoái hóa thần kinh."
 },
 {
  "id": "VI",
  "lang": "en",
  "text": "Diseases of the nervous system: headache, migraine, epilepsy, seizures, Parkinson disease, multiple sclerosis, meningitis, encephalitis, facial palsy, Bell palsy, sciatica, neuralgia, peripheral neuropathy, numbness and tingling in hands and feet, carpal tunnel syndrome, vestibular disorder, dizziness, vertigo, myasthenia gravis, cerebral palsy, sleep apnoea, neurodegenerative disease."
 },
 {
  "id": "VI",
  "lang": "zh",
  "text": "神经系统疾病：头痛、偏头痛、癫痫、抽搐、帕金森病、多发性硬化、脑膜炎、脑炎、面瘫、面神经麻痹、坐骨神经痛、神经痛、周围神经病变、手脚麻木、腕管综合征、前庭功能障碍、头晕、眩晕、重症肌无力、脑瘫、睡眠呼吸暂停、神经退行性疾病。"
 },
 {
  "id": "VII",
  "lang": "vi",
  "text": "Bệnh mắt và phần phụ của mắt: cận thị, viễn thị, loạn thị, lão thị, đục thủy tinh thể, cườm khô, glôcôm, cườm nước, tăng nhãn áp, viêm kết mạc, đau mắt đỏ, khô mắt, lẹo mắt, chắp mắt, viêm bờ mi, mộng mắt, bong võng mạc, thoái hóa điểm vàng, bệnh võng mạc tiểu đường, lác mắt, nhược thị, mỏi mắt, nhìn mờ, đeo kính, phẫu thuật mắt, mổ cận, nhỏ mắt."
 },
 {
  "id": "VII",
  "lang": "en",
  "text": "Diseases of the eye and adnexa: myopia, nearsightedness, hyperopia, astigmatism, presbyopia, cataract, glaucoma, raised intraocular pressure, conjunctivitis, pink eye, dry eye, stye, chalazion, blepharitis, pterygium, retinal detachment, macular degeneration, diabetic retinopathy, strabismus, squint, amblyopia, eye strain, blurred vision, glasses, eye surgery, LASIK, eye drops."
 },
 {
  "id": "VII",
  "lang": "zh",
  "text": "眼和附器疾病：近视、远视、散光、老花眼、白内障、青光眼、眼压高、结膜炎、红眼病、干眼症、麦粒肿、霰粒肿、睑缘炎、翼状胬肉、视网膜脱离、黄斑变性、糖尿病视网膜病变、斜视、弱视、视疲劳、视力模糊、眼镜、眼科手术、近视手术、眼药水。"
 },
 {
  "id": "VIII",
  "lang": "vi",
  "text": "Bệnh tai và xương chũm: viêm tai giữa, viêm tai ngoài, thủng màng nhĩ, ù tai, điếc, giảm thính lực, nghe kém, khiếm thính, viêm xương chũm, ráy tai, nút ráy tai, chảy mủ tai, đau tai, rối loạn tiền đình tai trong, bệnh Ménière, chóng mặt tư thế, máy trợ thính, cấy ốc tai."
 },
 {
  "id": "VIII",
  "lang": "en",
  "text": "Diseases of the ear and mastoid process: otitis media, middle ear infection, otitis externa, perforated eardrum, tinnitus, ringing in the ears, deafness, hearing loss, hard of hearing, mastoiditis, earwax, impacted cerumen, ear discharge, earache, inner ear vestibular disorder, Ménière disease, positional vertigo, hearing aids, cochlear implant."
 },
 {
  "id": "VIII",
  "lang": "zh",
  "text": "耳和乳突疾病：中耳炎、外耳道炎、鼓膜穿孔、耳鸣、耳聋、听力下降、听力障碍、乳突炎、耳垢、耵聍栓塞、耳朵流脓、耳痛、内耳前庭疾病、梅尼埃病、位置性眩晕、助听器、人工耳蜗。"
 },
 {
  "id": "IX",
  "lang": "vi",
  "text": "Bệnh hệ tuần hoàn, tim mạch: tăng huyết áp, cao huyết áp, huyết áp thấp, bệnh mạch vành, đau thắt ngực, nhồi máu cơ tim, suy tim, rối loạn nhịp tim, rung nhĩ, nhịp tim nhanh, hở van tim, hẹp van tim, bệnh cơ tim, viêm cơ tim, tim bẩm sinh, đột quỵ, tai biến mạch máu não, nhồi máu não, xuất huyết não, xơ vữa động mạch, giãn tĩnh mạch chân, suy giãn tĩnh mạch, huyết khối tĩnh mạch sâu, trĩ, phình động mạch. Điện tâm đồ, siêu âm tim, thuốc hạ huyết áp."
 },
 {
  "id": "IX",
  "lang": "en",
  "text": "Diseases of the circulatory system: hypertension, high blood pressure, hypotension, coronary artery disease, angina, myocardial infarction, heart attack, heart failure, arrhythmia, atrial fibrillation, tachycardia, heart valve regurgitation and stenosis, cardiomyopathy, myocarditis, congenital heart disease, stroke, cerebral infarction, brain haemorrhage, atherosclerosis, varicose veins, chronic venous insufficiency, deep vein thrombosis, haemorrhoids, aneurysm. ECG, echocardiography, antihypertensive drugs."
 },
 {
  "id": "IX",
  "lang": "zh",
  "text": "循环系统疾病：高血压、低血压、冠心病、心绞痛、心肌梗死、心力衰竭、心律失常、房颤、心动过速、心脏瓣膜关闭不全、瓣膜狭窄、心肌病、心肌炎、先天性心脏病、中风、脑卒中、脑梗死、脑出血、动脉粥样硬化、静脉曲张、深静脉血栓、痔疮、动脉瘤。心电图、心脏彩超、降压药。"
 },
 {
  "id": "X",
  "lang": "vi",
  "text": "Bệnh hệ hô hấp: cảm lạnh, cảm cúm, viêm mũi, viêm mũi dị ứng, viêm xoang, viêm họng, viêm amidan, viêm thanh quản, khàn tiếng, viêm phế quản, viêm tiểu phế quản, viêm phổi, hen suyễn, hen phế quản, bệnh phổi tắc nghẽn mạn tính COPD, giãn phế quản, tràn dịch màng phổi, tràn khí màng phổi, xơ phổi, ho kéo dài, ho có đờm, khó thở, thở khò khè, nghẹt mũi, sổ mũi, polyp mũi, vẹo vách ngăn, VA."
 },
 {
  "id": "X",
  "lang": "en",
  "text": "Diseases of the respiratory system: common cold, flu-like illness, rhinitis, allergic rhinitis, sinusitis, pharyngitis, sore throat, tonsillitis, laryngitis, hoarseness, bronchitis, bronchiolitis, pneumonia, asthma, chronic obstructive pulmonary disease COPD, bronchiectasis, pleural effusion, pneumothorax, pulmonary fibrosis, chronic cough, productive cough, shortness of breath, wheezing, nasal congestion, runny nose, nasal polyps, deviated septum, adenoids."
 },
 {
  "id": "X",
  "lang": "zh",
  "text": "呼吸系统疾病：感冒、鼻炎、过敏性鼻炎、鼻窦炎、咽炎、喉咙痛、扁桃体炎、喉炎、声音嘶哑、支气管炎、毛细支气管炎、肺炎、哮喘、慢性阻塞性肺病、支气管扩张、胸腔积液、气胸、肺纤维化、久咳、咳痰、呼吸困难、喘息、鼻塞、流鼻涕、鼻息肉、鼻中隔偏曲、腺样体肥大。"
 },
 {
  "id": "XI",
  "lang": "vi",
  "text": "Bệnh hệ tiêu hóa: sâu răng, viêm lợi, viêm nha chu, nhổ răng khôn, viêm loét miệng, trào ngược dạ dày thực quản, viêm thực quản, viêm dạ dày, loét dạ dày tá tràng, vi khuẩn HP, đau dạ dày, đầy hơi, khó tiêu, viêm đại tràng, hội chứng ruột kích thích, táo bón, tiêu chảy, viêm ruột thừa, thoát vị bẹn, sỏi mật, viêm túi mật, gan nhiễm mỡ, xơ gan, men gan cao, viêm tụy, polyp đại tràng, nứt hậu môn, rò hậu môn, xuất huyết tiêu hóa, nội soi dạ dày."
 },
 {
  "id": "XI",
  "lang": "en",
  "text": "Diseases of the digestive system: dental caries, tooth decay, gingivitis, periodontitis, wisdom tooth extraction, mouth ulcers, gastro-oesophageal reflux disease GERD, oesophagitis, gastritis, peptic and duodenal ulcer, Helicobacter pylori, stomach ache, bloating, indigestion, colitis, irritable bowel syndrome, constipation, diarrhoea, appendicitis, inguinal hernia, gallstones, cholecystitis, fatty liver, cirrhosis, raised liver enzymes, pancreatitis, colon polyps, anal fissure, anal fistula, gastrointestinal bleeding, gastroscopy."
 },
 {
  "id": "XI",
  "lang": "zh",
  "text": "消化系统疾病：龋齿、蛀牙、牙龈炎、牙周炎、拔智齿、口腔溃疡、胃食管反流、食管炎、胃炎、胃溃疡、十二指肠溃疡、幽门螺杆菌、胃痛、腹胀、消化不良、结肠炎、肠易激综合征、便秘、腹泻、阑尾炎、腹股沟疝、胆结石、胆囊炎、脂肪肝、肝硬化、转氨酶升高、胰腺炎、结肠息肉、肛裂、肛瘘、消化道出血、胃镜。"
 },
 {
  "id": "XII",
  "lang": "vi",
  "text": "Bệnh da và mô dưới da: mụn trứng cá, viêm da cơ địa, chàm, eczema, vảy nến, viêm da tiếp xúc, mề đay, dị ứng da, ngứa da, nổi mẩn đỏ, nám da, tàn nhang, sạm da, rụng tóc, hói đầu, gàu, viêm nang lông, áp xe da, mụn nhọt, chốc lở, bạch biến, sẹo lồi, rạn da, lang ben, nấm móng, móng chọc thịt, đổ mồ hôi nhiều, chăm sóc da, kem bôi da."
 },
 {
  "id": "XII",
  "lang": "en",
  "text": "Diseases of the skin and subcutaneous tissue: acne, atopic dermatitis, eczema, psoriasis, contact dermatitis, urticaria, hives, skin allergy, itchy skin, red rash, melasma, freckles, hyperpigmentation, hair loss, alopecia, dandruff, folliculitis, skin abscess, boils, impetigo, vitiligo, keloid scars, stretch marks, tinea versicolor, nail fungus, ingrown toenail, excessive sweating, skin care, topical creams."
 },
 {
  "id": "XII",
  "lang": "zh",
  "text": "皮肤和皮下组织疾病：痤疮、青春痘、特应性皮炎、湿疹、银屑病、牛皮癣、接触性皮炎、荨麻疹、皮肤过敏、皮肤瘙痒、红疹、黄褐斑、雀斑、色素沉着、脱发、斑秃、头皮屑、毛囊炎、皮肤脓肿、疖子、脓疱疮、白癜风、瘢痕疙瘩、妊娠纹、花斑癣、灰指甲、嵌甲、多汗症、皮肤护理、外用药膏。"
 },
 {
  "id": "XIII",
  "lang": "vi",
  "text": "Bệnh hệ cơ xương khớp và mô liên kết: thoái hóa khớp, thoái hóa cột sống, thoát vị đĩa đệm, đau lưng, đau cổ vai gáy, đau khớp gối, viêm khớp dạng thấp, viêm cột sống dính khớp, loãng xương, gai cột sống, cong vẹo cột sống, viêm gân, viêm bao hoạt dịch, đau cơ, chuột rút, lupus ban đỏ hệ thống, xơ cứng bì, đau xương, viêm khớp gout, bàn chân bẹt, vật lý trị liệu, phục hồi chức năng."
 },
 {
  "id": "XIII",
  "lang": "en",
  "text": "Diseases of the musculoskeletal system and connective tissue: osteoarthritis, spondylosis, herniated disc, back pain, low back pain, neck and shoulder pain, knee pain, rheumatoid arthritis, ankylosing spondylitis, osteoporosis, bone spurs, scoliosis, tendinitis, bursitis, muscle pain, cramps, systemic lupus erythematosus, scleroderma, bone pain, gouty arthritis, flat feet, physiotherapy, rehabilitation."
 },
 {
  "id": "XIII",
  "lang": "zh",
  "text": "肌肉骨骼系统和结缔组织疾病：骨关节炎、颈椎病、腰椎间盘突出、腰痛、颈肩痛、膝关节痛、类风湿关节炎、强直性脊柱炎、骨质疏松、骨刺、脊柱侧弯、肌腱炎、滑囊炎、肌肉疼痛、抽筋、系统性红斑狼疮、硬皮病、骨痛、痛风性关节炎、扁平足、物理治疗、康复训练。"
 },
 {
  "id": "XIV",
  "lang": "vi",
  "text": "Bệnh hệ tiết niệu sinh dục: sỏi thận, sỏi tiết niệu, viêm đường tiết niệu, viêm bàng quang, tiểu buốt, tiểu rắt, tiểu ra máu, suy thận, bệnh thận mạn, viêm cầu thận, hội chứng thận hư, chạy thận, tiểu đêm, tiểu không tự chủ, phì đại tuyến tiền liệt, viêm tuyến tiền liệt, vô sinh nam, yếu sinh lý, rối loạn cương dương, giãn tĩnh mạch thừng tinh, viêm âm đạo, viêm lộ tuyến cổ tử cung, rối loạn kinh nguyệt, đau bụng kinh, rong kinh, mãn kinh, u nang buồng trứng, lạc nội mạc tử cung, vô sinh nữ, bệnh tuyến vú."
 },
 {
  "id": "XIV",
  "lang": "en",
  "text": "Diseases of the genitourinary system: kidney stones, urinary stones, urinary tract infection, cystitis, painful urination, frequent urination, blood in urine, kidney failure, chronic kidney disease, glomerulonephritis, nephrotic syndrome, dialysis, nocturia, urinary incontinence, benign prostatic hyperplasia, prostatitis, male infertility, erectile dysfunction, varicocele, vaginitis, cervical ectropion, menstrual disorders, period pain, heavy menstrual bleeding, menopause, ovarian cyst, endometriosis, female infertility, breast disorders."
 },
 {
  "id": "XIV",
  "lang": "zh",
  "text": "泌尿生殖系统疾病：肾结石、尿路结石、尿路感染、膀胱炎、尿痛、尿频、血尿、肾衰竭、慢性肾病、肾小球肾炎、肾病综合征、透析、夜尿、尿失禁、前列腺增生、前列腺炎、男性不育、阳痿、勃起功能障碍、精索静脉曲张、阴道炎、宫颈糜烂、月经不调、痛经、月经过多、更年期、卵巢囊肿、子宫内膜异位症、女性不孕、乳腺疾病。"
 },
 {
  "id": "XV",
  "lang": "vi",
  "text": "Thai nghén, sinh đẻ và hậu sản: mang thai, có thai, thử thai, beta hCG, thai kỳ, khám thai, siêu âm thai, thai ngoài tử cung, sảy thai, dọa sảy thai, thai lưu, ốm nghén, tiền sản giật, tiểu đường thai kỳ, nhau tiền đạo, ra máu khi mang thai, sinh thường, sinh mổ, chuyển dạ, sinh non, băng huyết sau sinh, hậu sản, cho con bú, sữa mẹ, trầm cảm sau sinh, phá thai, kế hoạch hóa gia đình sau sinh."
 },
 {
  "id": "XV",
  "lang": "en",
  "text": "Pregnancy, childbirth and the puerperium: pregnancy, pregnancy test, beta hCG, gestation, antenatal check-up, prenatal ultrasound, ectopic pregnancy, miscarriage, threatened abortion, stillbirth, morning sickness, pre-eclampsia, gestational diabetes, placenta praevia, bleeding during pregnancy, vaginal delivery, caesarean section, labour, preterm birth, postpartum haemorrhage, postpartum period, breastfeeding, breast milk, postpartum depression, abortion."
 },
 {
  "id": "XV",
  "lang": "zh",
  "text": "妊娠、分娩和产褥期：怀孕、验孕、β-hCG、孕期、产检、孕期B超、宫外孕、流产、先兆流产、胎停、孕吐、子痫前期、妊娠糖尿病、前置胎盘、孕期出血、顺产、剖宫产、分娩、早产、产后出血、产褥期、母乳喂养、产后抑郁、人工流产。"
 },
 {
  "id": "XVI",
  "lang": "vi",
  "text": "Một số bệnh lý xuất phát trong thời kỳ chu sinh, trẻ sơ sinh: vàng da sơ sinh, trẻ sinh non, trẻ nhẹ cân, suy hô hấp sơ sinh, ngạt khi sinh, nhiễm trùng sơ sinh, hạ đường huyết sơ sinh, chăm sóc trẻ sơ sinh, rốn trẻ sơ sinh, trẻ sơ sinh bú kém, nôn trớ, quấy khóc ở trẻ sơ sinh, sang chấn khi sinh, lồng ấp."
 },
 {
  "id": "XVI",
  "lang": "en",
  "text": "Certain conditions originating in the perinatal period, newborn: neonatal jaundice, premature infant, low birth weight, neonatal respiratory distress, birth asphyxia, neonatal sepsis, neonatal hypoglycaemia, newborn care, umbilical cord care, poor feeding in newborns, spitting up, newborn crying, birth injury, incubator."
 },
 {
  "id": "XVI",
  "lang": "zh",
  "text": "起源于围生期的某些情况，新生儿：新生儿黄疸、早产儿、低出生体重、新生儿呼吸窘迫、出生窒息、新生儿败血症、新生儿低血糖、新生儿护理、脐带护理、新生儿吃奶差、吐奶、新生儿哭闹、产伤、保温箱。"
 },
 {
  "id": "XVII",
  "lang": "vi",
  "text": "Dị tật bẩm sinh, biến dạng và bất thường nhiễm sắc thể: hội chứng Down, bất thường nhiễm sắc thể, dị tật tim bẩm sinh, thông liên thất, thông liên nhĩ, còn ống động mạch, sứt môi hở hàm ếch, tật đầu nhỏ, não úng thủy, nứt đốt sống, tinh hoàn ẩn, lỗ tiểu lệch thấp, bàn chân khoèo, trật khớp háng bẩm sinh, thừa ngón, dính ngón, sàng lọc trước sinh, xét nghiệm NIPT, bệnh di truyền."
 },
 {
  "id": "XVII",
  "lang": "en",
  "text": "Congenital malformations, deformations and chromosomal abnormalities: Down syndrome, chromosomal abnormality, congenital heart defect, ventricular septal defect, atrial septal defect, patent ductus arteriosus, cleft lip and palate, microcephaly, hydrocephalus, spina bifida, undescended testis, hypospadias, clubfoot, congenital hip dysplasia, polydactyly, syndactyly, prenatal screening, NIPT, genetic disease."
 },
 {
  "id": "XVII",
  "lang": "zh",
  "text": "先天畸形、变形和染色体异常：唐氏综合征、染色体异常、先天性心脏病、室间隔缺损、房间隔缺损、动脉导管未闭、唇腭裂、小头畸形、脑积水、脊柱裂、隐睾、尿道下裂、马蹄内翻足、先天性髋关节发育不良、多指、并指、产前筛查、无创DNA、遗传病。"
 },
 {
  "id": "XVIII",
  "lang": "vi",
  "text": "Triệu chứng, dấu hiệu và kết quả bất thường chưa phân loại: sốt, sốt cao, mệt mỏi, chán ăn, sụt cân, đau bụng, buồn nôn, nôn, khó nuốt, nghẹn, đau ngực, khó thở, ho, chóng mặt, ngất, phù, nổi hạch, đau khắp người, ra mồ hôi đêm, tiểu ra máu không rõ nguyên nhân, kết quả xét nghiệm bất thường, chỉ số xét nghiệm máu, siêu âm phát hiện bất thường, triệu chứng là bệnh gì, có nguy hiểm không."
 },
 {
  "id": "XVIII",
  "lang": "en",
  "text": "Symptoms, signs and abnormal findings not elsewhere classified: fever, high fever, fatigue, loss of appetite, weight loss, abdominal pain, nausea, vomiting, difficulty swallowing, choking feeling, chest pain, shortness of breath, cough, dizziness, fainting, oedema, swollen lymph nodes, body aches, night sweats, unexplained blood in urine, abnormal laboratory results, blood test values, abnormal ultrasound finding, what disease causes this symptom, is it dangerous."
 },
 {
  "id": "XVIII",
  "lang": "zh",
  "text": "症状、体征和异常发现：发烧、高烧、乏力、食欲不振、体重下降、腹痛、恶心、呕吐、吞咽困难、哽噎感、胸痛、呼吸困难、咳嗽、头晕、晕厥、水肿、淋巴结肿大、全身疼痛、盗汗、不明原因血尿、化验结果异常、血液检查指标、B超异常、这是什么病、严重吗。"
 },
 {
  "id": "XIX",
  "lang": "vi",
  "text": "Chấn thương, ngộ độc và hậu quả của nguyên nhân bên ngoài: gãy xương, bong gân, trật khớp, đứt dây chằng, chấn thương đầu, chấn thương sọ não, vết thương, vết cắt, bỏng, bỏng nước sôi, say nắng, sốc nhiệt, ngộ độc thực phẩm, ngộ độc thuốc, quá liều, rắn cắn, chó cắn, côn trùng đốt, dị vật, hóc xương, đuối nước, điện giật, sốc phản vệ, dị ứng thuốc, phản ứng sau tiêm, biến chứng sau phẫu thuật, sơ cứu."
 },
 {
  "id": "XIX",
  "lang": "en",
  "text": "Injury, poisoning and certain other consequences of external causes: fractures, broken bone, sprain, dislocation, torn ligament, head injury, traumatic brain injury, wounds, cuts, burns, scalds, heat stroke, food poisoning, drug poisoning, overdose, snake bite, dog bite, insect sting, foreign body, choking on a bone, drowning, electric shock, anaphylaxis, drug allergy, adverse reaction after injection, postoperative complications, first aid."
 },
 {
  "id": "XIX",
  "lang": "zh",
  "text": "损伤、中毒和外因的某些其他后果：骨折、扭伤、脱臼、韧带撕裂、头部外伤、颅脑损伤、伤口、割伤、烧伤、烫伤、中暑、食物中毒、药物中毒、用药过量、蛇咬伤、狗咬伤、蚊虫叮咬、异物、鱼刺卡喉、溺水、触电、过敏性休克、药物过敏、注射后不良反应、术后并发症、急救。"
 },
 {
  "id": "XX",
  "lang": "vi",
  "text": "Nguyên nhân bên ngoài gây bệnh tật và tử vong: tai nạn giao thông, tai nạn xe máy, tai nạn lao động, ngã, té ngã ở người già, tai nạn sinh hoạt, bạo lực, hành hung, tự gây thương tích, tác dụng phụ của thuốc khi dùng đúng liều, biến cố y khoa, sai sót trong điều trị, phơi nhiễm hóa chất, ô nhiễm môi trường, tiếp xúc khói bụi."
 },
 {
  "id": "XX",
  "lang": "en",
  "text": "External causes of morbidity and mortality: traffic accidents, motorbike accidents, occupational accidents, falls, falls in the elderly, household accidents, violence, assault, self-harm, adverse effects of drugs in correct use, medical incidents, treatment errors, chemical exposure, environmental pollution, smoke and dust exposure."
 },
 {
  "id": "XX",
  "lang": "zh",
  "text": "疾病和死亡的外因：交通事故、摩托车事故、工伤事故、跌倒、老年人跌倒、家庭意外、暴力、伤害、自残、药物正常使用时的不良反应、医疗事件、治疗差错、化学品暴露、环境污染、烟尘暴露。"
 },
 {
  "id": "XXI",
  "lang": "vi",
  "text": "Các yếu tố ảnh hưởng đến sức khỏe và tiếp xúc dịch vụ y tế: khám sức khỏe định kỳ, tầm soát, xét nghiệm tổng quát, tiêm chủng, lịch tiêm vắc xin cho trẻ, tư vấn dinh dưỡng, chế độ ăn uống, tập luyện, giảm cân, chăm sóc sức khỏe người cao tuổi, sức khỏe sinh sản, tránh thai, thuốc tránh thai, đặt vòng, tư vấn tiền hôn nhân, hỗ trợ sinh sản, thụ tinh ống nghiệm IVF, bảo hiểm y tế, đặt lịch khám, thủ tục bệnh viện, chăm sóc sau phẫu thuật, phục hồi."
 },
 {
  "id": "XXI",
  "lang": "en",
  "text": "Factors influencing health status and contact with health services: routine health check-up, screening, general tests, vaccination, childhood immunisation schedule, nutrition counselling, diet, exercise, weight loss, elderly care, reproductive health, contraception, birth control pills, IUD, premarital counselling, assisted reproduction, in vitro fertilisation IVF, health insurance, booking an appointment, hospital procedures, aftercare following surgery, recovery."
 },
 {
  "id": "XXI",
  "lang": "zh",
  "text": "影响健康状态和与保健机构接触的因素：定期体检、筛查、常规检查、疫苗接种、儿童免疫程序、营养咨询、饮食、锻炼、减肥、老年保健、生殖健康、避孕、避孕药、上环、婚前检查、辅助生殖、试管婴儿、医疗保险、预约挂号、医院流程、术后护理、康复。"
 },
 {
  "id": "XXII",
  "lang": "vi",
  "text": "Mã cho mục đích đặc biệt: COVID-19, SARS-CoV-2, nhiễm virus corona, hậu COVID, Long COVID, tiêm vắc xin phòng COVID-19, mũi tiêm nhắc lại, chống chỉ định tiêm vắc xin COVID, phản ứng sau tiêm vắc xin COVID, test nhanh kháng nguyên, xét nghiệm PCR, cách ly, F0, F1, bệnh mới nổi chưa rõ nguyên nhân, kháng thuốc kháng sinh, vi khuẩn kháng thuốc."
 },
 {
  "id": "XXII",
  "lang": "en",
  "text": "Codes for special purposes: COVID-19, SARS-CoV-2, coronavirus infection, post-COVID condition, long COVID, COVID-19 vaccination, booster dose, contraindications to COVID vaccines, reactions after COVID vaccination, rapid antigen test, PCR test, quarantine, isolation, emerging diseases of uncertain aetiology, antimicrobial resistance, drug-resistant bacteria."
 },
 {
  "id": "XXII",
  "lang": "zh",
  "text": "特殊目的编码：新冠肺炎、新型冠状病毒、冠状病毒感染、新冠后遗症、长新冠、新冠疫苗接种、加强针、新冠疫苗禁忌、新冠疫苗接种后反应、抗原快速检测、核酸检测、隔离、病因不明的新发疾病、抗生素耐药、耐药菌。"
 }
]

In [ ]:
import subprocess

import torch

print(subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total,driver_version', '--format=csv'],
                     capture_output=True, text=True).stdout)
GPU_NAME = torch.cuda.get_device_name(0)
assert EXPECT_GPU in GPU_NAME, f'expected a GPU named *{EXPECT_GPU}*, got {GPU_NAME!r}; stop here'
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
print('GPU:', GPU_NAME, '| dtype:', DTYPE, '| torch', torch.__version__)

In [ ]:
from transformers import AutoModel, AutoTokenizer

os.environ['HF_HUB_OFFLINE'] = '1'
hits = sorted(glob.glob(f'/kaggle/input/**/{MODEL_DIR}/config.json', recursive=True))
assert hits, f'attach boinhbo/medxir-package-models (kernel source) with {MODEL_DIR} before Run All'
SRC = os.path.dirname(hits[0])
start = time.perf_counter()
tok = AutoTokenizer.from_pretrained(SRC)
tok.padding_side = 'left'
model = AutoModel.from_pretrained(SRC, torch_dtype=DTYPE, attn_implementation='sdpa').cuda().eval()
LOAD_S = round(time.perf_counter() - start, 1)
print('loaded', SRC, 'in', LOAD_S, 's | hidden size', model.config.hidden_size)

In [ ]:
import numpy as np
import torch.nn.functional as F


def last_token_pool(hidden, mask):
    """Last-token pooling from the Qwen3-Embedding model card (handles left and right padding)."""
    if mask[:, -1].sum() == mask.shape[0]:
        return hidden[:, -1]
    lengths = mask.sum(dim=1) - 1
    return hidden[torch.arange(hidden.shape[0], device=hidden.device), lengths]


@torch.inference_mode()
def embed_texts(texts):
    """Float16 unit vectors in the input order, and the number of tokens processed."""
    ids = tok(texts, truncation=True, max_length=MAX_LENGTH)['input_ids']
    order = np.argsort([-len(x) for x in ids], kind='stable')  # longest first: less padding
    out = np.empty((len(texts), model.config.hidden_size), dtype=np.float16)
    for start in range(0, len(order), BATCH):
        index = order[start:start + BATCH]
        enc = tok.pad({'input_ids': [ids[i] for i in index]}, return_tensors='pt').to('cuda')
        hidden = model(input_ids=enc['input_ids'], attention_mask=enc['attention_mask'], use_cache=False).last_hidden_state
        vec = F.normalize(last_token_pool(hidden, enc['attention_mask']).float(), p=2, dim=1)
        out[index] = vec.cpu().numpy().astype(np.float16)
    return out, sum(len(x) for x in ids)

In [ ]:
vectors, tokens = embed_texts([DOC_PREFIX + row['text'] for row in ROWS])
assert vectors.shape == (len(ROWS), 4096) and np.isfinite(vectors.astype(np.float32)).all()
np.save(f'{OUT}/icd_vectors.npy', vectors)
with open(f'{OUT}/icd_rows.json', 'w', encoding='utf-8') as file:
    json.dump([{k: row[k] for k in ('id', 'lang')} for row in ROWS], file, ensure_ascii=False)
print('embedded', len(ROWS), 'texts,', tokens, 'tokens')